In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import time
import os
from IPython.display import clear_output, HTML
import matplotlib.animation as animation
# from pyDOE import lhs
from torch.func import vmap, jacrev
from copy import deepcopy


from config import *
from grid import gLLNodesAndWeights, lgP
from data_generator2 import u_train_data_generator#, test_data_generator
from diff import DiffMat
torch.cuda.empty_cache()
torch.set_default_dtype(torch.float64)

print(device)

np.random.seed(1234)

In [ ]:
start_time = time.perf_counter()
Folder = 'outputs/training' if TRAIN else f'Models_PDEloss_{model_case[0][1][0]}D'
if TRAIN:
    file_path = os.path.join(f'{Folder}', "info.txt")
    os.makedirs(f'{Folder}', exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as tmp:
        for i in range(len(model_case)):
            tmp.write(f'PINN_model_{i}: Ngrid={model_case[i][0]}, net_dim={model_case[i][1]}, act.={model_case[i][2]}, FFsize={model_case[i][3]}, FFscale={model_case[i][4]}, mu={model_case[i][5]}\n')


In [ ]:
def save_model(model, folder, filename="model.pth", save_entire=False):
    """
    儲存模型到指定資料夾
    save_entire = False -> 只存參數 (建議用這個，安全且通用)
    save_entire = True  -> 存整個模型 (包含架構)
    """
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, filename)

    if save_entire:
        torch.save(model, path)  # 存整個模型 (包含架構)
        print(f"完整模型已儲存到 {path}")
    else:
        torch.save(model.state_dict(), path)  # 只存參數
        print(f"模型參數已儲存到 {path}")


def load_model(model_class, folder, filename="model.pth", device="cpu", load_entire=False, layout=True):
    """
    讀取模型
    load_entire = False -> 載入參數，需要傳入 model_class
    load_entire = True  -> 直接載入完整模型，不需要 model_class
    """
    path = os.path.join(folder, filename)

    if load_entire:
        model = torch.load(path, map_location=device, weights_only=False)
        if layout:
            print(f"完整模型已從 {path} 載入")
    else:
        model = model_class().to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=False))
        model.eval()
        print(f"模型參數已從 {path} 載入")

    return model

In [ ]:
from deepsplines.ds_modules import dsnn

class FourierFeatures(nn.Module):
    def __init__(self, in_dim, mapping_size=64, scale=5.0, mu=5):
        """
        in_dim: 輸入維度，例如 G(x,y,s,t) 就是 4
        mapping_size: Fourier features 的頻率數量
        scale: 頻率尺度，越大代表越高頻
        """
        g = torch.Generator(device=device)
        g.manual_seed(1234)

        super().__init__()
        # B = torch.randn(in_dim, mapping_size, generator=g, device=device) * scale
        
        if mapping_size %2==0:
            B1 = torch.randn(in_dim, mapping_size//2, generator=g, device=device)* scale + mu
        else:
            B1 = torch.randn(in_dim, mapping_size//2 + 1, generator=g, device=device)* scale + mu
        B2 = torch.randn(in_dim, mapping_size//2, generator=g, device=device)* scale - mu
        B = torch.cat([B1, B2], dim=-1)

        # B = generate_matrix(mapping_size)
        self.register_buffer("B", B)

    def forward(self, x):
        """
        x: (..., in_dim)
        return: (..., 2 * mapping_size)
        """
        x_proj = 2 * torch.pi * x @ self.B   # (..., mapping_size)
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)
    
class NeuralNet_4D(dsnn.DSModule, nn.Module):
    def __init__(self, net_dim, act=nn.ReLU(), use_norm=False, mapping_size=64, scale=5.0, use_fourier=True, mu=5):
        super(NeuralNet_4D, self).__init__()

        self.use_fourier = use_fourier
        self.use_norm = use_norm
        self.act = act
        self.layers = nn.ModuleList()
        self.fc_ds = nn.ModuleList()
        self.norms = nn.ModuleList()

        # deepspline parameters
        opt_params = {
            'size': 21,
            'range_': 4,
            'init': 'leaky_relu',
            'save_memory': False
        }
        
        input_dim = net_dim[0]

        # Fourier Features
        if self.use_fourier:
            self.ff = FourierFeatures(in_dim=input_dim, mapping_size=mapping_size, scale=scale, mu=mu).to(device)
            input_dim = 2 * mapping_size # Fourier 後的實際輸入維度
            # input_dim = 2 * ((2*mapping_size+1)**4)  
        else:
            self.ff = None

        dims = [input_dim] + net_dim[1:]

        self.depth = len(dims) - 1

        for i in range(self.depth - 1):
            layer = nn.Linear(dims[i], dims[i+1])
            self.layers.append(layer)
            
            if self.act == 'dsnn':
                self.fc_ds.append(dsnn.DeepBSpline('fc', dims[i+1], **opt_params))
            if self.use_norm:
                norm = nn.LayerNorm(dims[i+1])
                self.norms.append(norm)

        self.final = nn.Linear(dims[-2], dims[-1])

    def forward(self, x):
        if self.use_fourier:
            x = self.ff(x)
        for i in range(self.depth - 1):
            x = self.layers[i](x)
            if self.use_norm:
                x = self.norms[i](x)

            if self.act == 'dsnn':
                x = self.fc_ds[i](x)
            else:
                x = self.act(x)

        return self.final(x)

In [ ]:
# from deepsplines.ds_modules import dsnn

# class FourierFeatures(nn.Module):
#     def __init__(self, in_dim, mapping_size=64, scale=5.0, mu=5):
#         """
#         in_dim: 輸入維度，例如 G(x,y,s,t) 就是 4
#         mapping_size: Fourier features 的頻率數量
#         scale: 頻率尺度，越大代表越高頻
#         """
#         g = torch.Generator(device=device)
#         g.manual_seed(1234)

#         super().__init__()
#         # B = torch.randn(in_dim, mapping_size, generator=g, device=device) * scale
        
#         if mapping_size %2==0:
#             B1 = torch.randn(in_dim, mapping_size//2, generator=g, device=device)* scale + mu
#         else:
#             B1 = torch.randn(in_dim, mapping_size//2 + 1, generator=g, device=device)* scale + mu
#         B2 = torch.randn(in_dim, mapping_size//2, generator=g, device=device)* scale - mu
#         B = torch.cat([B1, B2], dim=-1)

#         self.register_buffer("B", B)

#     def forward(self, x):
#         """
#         x: (..., in_dim)
#         return: (..., 2 * mapping_size)
#         """

#         x_proj = 2 * torch.pi * x @ self.B   # (..., mapping_size)
#         return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)
    
# class NeuralNet(dsnn.DSModule, nn.Module):
#     def __init__(self, net_dim, act=nn.ReLU(), use_norm=False, mapping_size=64, scale=5.0, use_fourier=False, mu=5):
#         super(NeuralNet, self).__init__()

#         self.use_fourier = use_fourier
#         self.use_norm = use_norm
#         self.act = act
#         self.layers = nn.ModuleList()
#         self.sublayers_xy = nn.ModuleList()
#         self.sublayers_st = nn.ModuleList()
#         self.fc_ds = nn.ModuleList()
#         self.norms = nn.ModuleList()

#         # deepspline parameters
#         opt_params = {
#             'size': 21,
#             'range_': 4,
#             'init': 'leaky_relu',
#             'save_memory': False
#         }
        
#         input_dim = net_dim[0]
#         sub_dim = net_dim[1]
        
#         # Fourier Features
#         input_dim = net_dim[0]

#         sub_dim_xy = net_dim[1][0]
#         sub_dim_st = net_dim[1][1]

#         if self.use_fourier:
#             self.ff = FourierFeatures(in_dim=input_dim,mapping_size=mapping_size,scale=scale,mu=mu)
#             ff_dim = 2 * mapping_size
#         else:
#             ff_dim = input_dim

#         dims = [input_dim] + net_dim[1:]
#         self.depth = len(dims) - 1

#         # xy, st各自子網路
#         self.sublayers_xy = nn.Linear(ff_dim, sub_dim_xy)
#         self.sublayers_st = nn.Linear(ff_dim, sub_dim_st)

#         # 合併xy, st到主網路
#         main_dims = [sub_dim_xy + sub_dim_st, *net_dim[2:]]
#         for i in range(len(main_dims)-1):

#             self.layers.append(nn.Linear(main_dims[i], main_dims[i+1]))

#             if i < len(main_dims)-2:
#                 if self.use_norm:
#                     self.norms.append(nn.LayerNorm(main_dims[i+1]))

#                 if self.act == 'dsnn':
#                     self.fc_ds.append(dsnn.DeepBSpline('fc', main_dims[i+1], **opt_params))

#         self.final = nn.Linear(dims[-2], dims[-1])

#     def forward(self, xy, st):

#         if self.use_fourier:
#             xy = self.ff(xy)
#             st = self.ff(st)

#         xy = self.act(self.sublayers_xy(xy))

#         st = self.act(self.sublayers_st(st))
#         x = torch.cat([xy, st], dim=-1)
        
#         for i, layer in enumerate(self.layers[:-1]):
#             x = layer(x)
#             if self.use_norm:
#                 x = self.norms[i](x)
                
#             if self.act == 'dsnn':
#                 x = self.fc_ds[i](x)
#             else:
#                 x = self.act(x)

#         x = self.layers[-1](x)

#         return x

In [ ]:
class NeuralNet_2D(dsnn.DSModule, nn.Module):
    def __init__(self, net_dim, act=nn.ReLU(), use_norm=False, mapping_size=64, scale=5.0, use_fourier=False, mu=5):
        super(NeuralNet_2D, self).__init__()

        self.use_fourier = use_fourier
        self.use_norm = use_norm
        self.act = act
        self.Rank = net_dim[-1]  # 多秩近似的數量
        net_dim = net_dim[:-1]
        
        # deepspline 參數
        opt_params = {'size': 21, 'range_': 4, 'init': 'leaky_relu', 'save_memory': False}
        
        input_dim = net_dim[0]
        sub_dim = net_dim[1] # 假設 net_dim[1] 是一個整數，例如 128
        
        if self.use_fourier:
            self.ff = FourierFeatures(in_dim=input_dim, mapping_size=mapping_size, scale=scale, mu=mu)
            ff_dim = 2 * mapping_size
        else:
            ff_dim = input_dim

        # ---- 建構 xy 專屬的獨立子網路 (Branch Net) ----
        self.xy_layers = nn.ModuleList()
        self.xy_norms = nn.ModuleList()
        if self.act == 'dsnn': self.xy_ds = nn.ModuleList()
        
        # ---- 建構 st 專屬的獨立子網路 (Trunk Net) ----
        self.st_layers = nn.ModuleList()
        self.st_norms = nn.ModuleList()
        if self.act == 'dsnn': self.st_ds = nn.ModuleList()

        # 這裡的 net_dim[1:] 包含隱層維度，例如 [128, 128, 128]
        dims = [ff_dim] + list(net_dim[1:])
        
        for i in range(len(dims) - 1):
            # 兩套完全獨立的線性層
            self.xy_layers.append(nn.Linear(dims[i], dims[i+1]))
            self.st_layers.append(nn.Linear(dims[i], dims[i+1]))
            
            if i < len(dims) - 1:
                if self.use_norm:
                    self.xy_norms.append(nn.LayerNorm(dims[i+1]))
                    self.st_norms.append(nn.LayerNorm(dims[i+1]))
                if self.act == 'dsnn':
                    self.xy_ds.append(dsnn.DeepBSpline('fc', dims[i+1], **opt_params))
                    self.st_ds.append(dsnn.DeepBSpline('fc', dims[i+1], **opt_params))

        # ---- 【關鍵】最後一層各自輸出 Rank 個通道，完全獨立 ----
        self.xy_final = nn.Linear(dims[-1], self.Rank)
        self.st_final = nn.Linear(dims[-1], self.Rank)

    def forward(self, xy, st):
        # 1. 2D 傅立葉映射
        if self.use_fourier:
            xy = self.ff(xy)
            st = self.ff(st)

        # 2. xy 獨立通過自己的網路
        for i, layer in enumerate(self.xy_layers):
            xy = layer(xy)
            if self.use_norm: xy = self.xy_norms[i](xy)
            if self.act == 'dsnn': xy = self.xy_ds[i](xy)
            else: xy = self.act(xy)
        phi_rank = self.xy_final(xy) # (Batch, Ngrid+1, Ngrid+1, Rank)

        # 3. st 獨立通過自己的網路
        for i, layer in enumerate(self.st_layers):
            st = layer(st)
            if self.use_norm: st = self.st_norms[i](st)
            if self.act == 'dsnn': st = self.st_ds[i](st)
            else: st = self.act(st)
        psi_rank = self.st_final(st) # (Batch, Ngrid+1, Ngrid+1, Rank)

        # 4. 疊加
        phi_exp = phi_rank.unsqueeze(3).unsqueeze(4) # (B, N, N, 1, 1, Rank)
        psi_exp = psi_rank.unsqueeze(1).unsqueeze(2) # (B, 1, 1, N, N, Rank)
        
        G_pde = torch.sum(phi_exp * psi_exp, dim=-1) # (B, N, N, N, N)

        return G_pde

In [ ]:
case = 0
G_Linf = []
u_Linf_num = []
u_Linf_exact = []
u_L2_exact = []
for Ngrid, net_dim, act, FF_size, FF_scale, mu in model_case:
    
    train_start_time = time.perf_counter()
    model_folder = f'{Folder}/PINN_model_{case}'
    output_folder = model_folder if TRAIN else f'outputs/evaluation/PINN_model_{case}'
    os.makedirs(f'{output_folder}/imgs', exist_ok=True)

    if net_dim[0] == 2:
        nnFuncApprox = NeuralNet_2D(net_dim, act, mapping_size=FF_size, scale=FF_scale,
                                use_fourier=USE_FF, mu=mu, use_norm=USE_NORM).to(device)
        
    elif net_dim[0] == 4:
        nnFuncApprox = NeuralNet_4D(net_dim, act, mapping_size=FF_size, scale=FF_scale,
                                use_fourier=USE_FF, mu=mu, use_norm=USE_NORM).to(device)
    params = dict(nnFuncApprox.named_parameters())
    par_num = 0
    for par in params:
        par_num += torch.numel(params[par])
    print(f'Number of overall trainable parameters = {par_num}')
    print(nnFuncApprox)

    file_path = os.path.join(output_folder, "info.txt")
    with open(file_path, "w", encoding="utf-8") as tmp:
        tmp.write(f'net_dim={net_dim}, act.={act}, FFsize={FF_size}, FFscale={FF_scale}, mu={mu}\n')
        tmp.write(f'Ngrid={Ngrid}\n')
        tmp.write(str(nnFuncApprox))

    optimizerAdam = torch.optim.AdamW(nnFuncApprox.parameters(), lr=0.001, weight_decay=1e-6)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizerAdam, 
                    T_max=Adam_iter,    # 總步數
                    eta_min=1e-5        # 降到最後的最低學習率
                )
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    #                 optimizerAdam, 
    #                 T_0=Change_data,     # 第一個週期的步數
    #                 T_mult=1,     # 後續週期的倍數（1代表每個週期都固定是 2000 步）
    #                 eta_min=1e-5  # 每個週期谷底的最低學習率
    #             )

    # scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizerAdam,
    #                                                         mode='min',
    #                                                         factor=0.5,
    #                                                         patience=1000,
    #                                                         threshold=1e-2,
    #                                                         threshold_mode='rel',
    #                                                         cooldown=0,
    #                                                         min_lr=1e-4,
    #                                                         eps=1e-12
    #                                                     )
    if TRAIN:
        savedloss    = [1e16]
        savedloss_ts = [1e16]
        
        nnFuncApprox.train()
        best_loss = 1e16
        patience_counter = 0
        # 固定生成100筆 validattion data
        validate_set_size = Validate_Batch_size
        with torch.no_grad():
            _, validate_data, validate_data_xy, validate_data_st, validate_label_f, validate_label_u = u_train_data_generator(Ngrid, validate_set_size) 
            validate_data_xy = validate_data_xy.to(device)
            validate_data_st = validate_data_st.to(device)
            validate_data    = validate_data.to(device)
            validate_label_f = validate_label_f.to(device)
            validate_label_u = validate_label_u.to(device)

        for step in range(Adam_iter):
            if step % Change_data == 0:
                a, b, c = torch.distributions.Uniform(-1,1).sample((1,1)).item(), torch.distributions.Uniform(-1,1).sample((1,1)).item(), torch.distributions.Uniform(-1,1).sample((1,1)).item()
                ((X2d, Y2d), (wx, wy), W2d, D, D2, U, F), train_data, train_data_xy, train_data_st, train_label_f, train_label_u = u_train_data_generator(Ngrid, Batch_Size)
                train_data_xy = train_data_xy.to(device)
                train_data_st = train_data_st.to(device)
                train_data    = train_data.to(device)
                train_label_f = train_label_f.to(device)
                train_label_u = train_label_u.to(device)
                D2 = D2.to(device)
                W2d = W2d.to(device)
            # Backward and optimize    
            optimizerAdam.zero_grad()

            if net_dim[0] == 2:
                G_pred = nnFuncApprox(train_data_xy, train_data_st)
            elif net_dim[0] == 4:
                G_pred = nnFuncApprox(train_data)
            G_reshape = G_pred.reshape(Batch_Size, Ngrid+1, Ngrid+1, Ngrid+1, Ngrid+1)
            
            wF = torch.einsum('bij,ij->bij', train_label_f, W2d)
            u_pred = torch.einsum('bijkl,bkl->bij', G_reshape, wF)
            r1 = torch.abs(u_pred - train_label_u) # r1 = G*wF - U
            r2 = torch.abs((D2@u_pred + u_pred@D2.T) - train_label_f) # r2 = \nabla U - F

            r = r1**2 + Lambda * r2**2
            lossAdam = (r).mean()
            savedloss.append(lossAdam.item())
            lossAdam.backward()
            
            optimizerAdam.step()
            scheduler.step()
            current_lr = optimizerAdam.param_groups[0]['lr']
            current_loss = lossAdam.item()

            savedloss_ts.append(savedloss_ts[-1])

            if lossAdam.item() < best_loss:
                best_loss = lossAdam.item()
                best_state_dict = deepcopy(nnFuncApprox.state_dict())

            if lossAdam.item() <= TOLERANCE:
                print(f"\nEarly stopping triggered at step {step} due to low loss.")
                print(f"Best loss: {best_loss:.6e}")
                break
            
            if step % 100 == 0:
                # Validation
                with torch.no_grad():
                    if net_dim[0] == 2:
                        G_validate = nnFuncApprox(validate_data_xy, validate_data_st)
                    elif net_dim[0] == 4:
                        G_validate = nnFuncApprox(validate_data)
                    G_validate_reshape = G_validate.reshape(validate_set_size, Ngrid+1, Ngrid+1, Ngrid+1, Ngrid+1)
                    wF_validate = torch.einsum('bij,ij->bij', validate_label_f, W2d)
                    u_validate = torch.einsum('bijkl,bkl->bij', G_validate_reshape, wF_validate)
                    r1 = torch.abs(u_validate - validate_label_u)
                    r2 = torch.abs((D2@u_validate + u_validate@D2.T) - validate_label_f)
                
                    r = r1**2 + Lambda * r2**2
                    valid_loss = (r).mean()
                    
                    savedloss_ts.append(valid_loss.item())
            
                print(f'Epoch {step}, LR={current_lr:.2e}, Training_Loss={lossAdam.item():.4e}, Validation_Loss={valid_loss.item():.4e}')
                print(f'Data Loss (r1): {r1.mean().item():.4e} | PDE Loss (r2): {r2.mean().item():.4e}')
        nnFuncApprox.load_state_dict(best_state_dict)
        save_model(nnFuncApprox, f'{model_folder}/models', filename=f"model_single.pth", save_entire=True)
        
        ########################################## plot #############################################
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        ax.plot(np.linspace(0, len(savedloss)-1, len(savedloss)-1), savedloss[1:], label='training loss')
        ax.plot(np.linspace(0, len(savedloss_ts)-1, len(savedloss_ts)-1), savedloss_ts[1:], label='validation loss')
        ax.set_yscale('log')
        ax.set_title(f'Loss of training, FF_size: {FF_size}, FF_scale: {FF_scale}, mu: {mu}')
        ax.set_ylim((min(savedloss)-10, max(savedloss[1:])+10))
        ax.set_xlabel('epochs')
        ax.legend()
        fname = f'{output_folder}/imgs/loss_1.jpg'
        plt.savefig(fname)
        plt.close()
        

    ########################################## predict #############################################
    with torch.no_grad():
        ((X2d, Y2d), (wx, wy), W2d, D, D2, U, F), train_data, train_data_xy, train_data_st, train_label_f, train_label_u = u_train_data_generator(Ngrid, 1)
        train_data_xy = train_data_xy.to(device)
        train_data_st = train_data_st.to(device)
        train_data    = train_data.to(device)
        train_label_f = train_label_f.to(device)
        train_label_u = train_label_u.to(device)
        
        loaded_model = load_model(None, f'{model_folder}/models', f"model_single.pth", device=f'{device}', load_entire=True)
        if net_dim[0] == 2:
            G_pred = loaded_model(train_data_xy, train_data_st).reshape(Ngrid+1, Ngrid+1, Ngrid+1, Ngrid+1).cpu()
        elif net_dim[0] == 4:
            G_pred = loaded_model(train_data).reshape(Ngrid+1, Ngrid+1, Ngrid+1, Ngrid+1).cpu()
        
        # r = (G_pred - train_label.reshape(Ngrid+1, Ngrid+1, Ngrid+1, Ngrid+1).cpu()).abs()  
        # G_Linf.append(torch.max(r).item())

        U = u(X2d, Y2d)
        F = D2@U +U@D2.T
        u_pred = torch.einsum('ijkl,kl->ij', G_pred, W2d * F)
        u_residual_exact = u_pred - U
        err_exact = torch.max(torch.abs(u_residual_exact))
        u_Linf_exact.append(err_exact.item())
        print(f'Maximum Error (vs exact): {err_exact.item():.2e}')

        
    # u_residual_num = u_pred - U
    # err_num = torch.max(torch.abs(u_residual_num))
    # u_Linf_num.append(err_num.item())
    # print(f'Maximum Error (vs numerical): {err_num.item():.2e}')
    
    # plt.matshow(r)
    # plt.title(f'Point-wise Error(G_NN-G_num),\n FF_size: {FF_size}, FF_scale: {FF_scale}, mu: {mu}')
    # plt.xlabel(r'$y_j, t_\ell$')
    # plt.ylabel(r'$x_i, s_k$')
    # plt.colorbar()
    # fname = f'{output_folder}/imgs/points_residual.jpg'
    # plt.savefig(fname)
    # plt.close()

    fig = plt.figure(figsize=(16, 8))
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(X2d.cpu(), Y2d.cpu(), U.cpu(), alpha=0.6)
    ax1.set_title('exact u')

    ax2 = fig.add_subplot(122, projection='3d')
    ax2.plot_surface(X2d.cpu(), Y2d.cpu(), u_pred.cpu(), alpha=0.6)
    ax2.set_title('preidict u')

    fname = f'{output_folder}/imgs/exact_u_and_predict_u.jpg'
    plt.savefig(fname)
    #plt.close()

    plt.matshow(u_residual_exact.abs().cpu().detach().numpy())
    plt.title(f'Point-wise Error(u_NN - u_exact),\n FF_size: {FF_size}, FF_scale: {FF_scale}, mu: {mu}')
    plt.xlabel(r'$y_j$')
    plt.ylabel(r'$x_i$')
    plt.colorbar()
    fname = f'{output_folder}/imgs/u_points_residual_exact.jpg'
    plt.savefig(fname)
    #plt.close()

    # plt.matshow(u_residual_num.abs().cpu().detach().numpy())
    # plt.title(f'Point-wise Error(u_NN - u_num),\n FF_size: {FF_size}, FF_scale: {FF_scale}, mu={mu}')
    # plt.xlabel(r'$y_j$')
    # plt.ylabel(r'$x_i$')
    # plt.colorbar()
    # fname = f'{output_folder}/imgs/u_points_residual_num.jpg'
    # plt.savefig(fname)
    # plt.close()

    end_time = time.perf_counter()
    run_time = end_time - train_start_time
    
    case += 1


In [ ]:
# a, b, c = torch.distributions.Uniform(1,2).sample((1,1)).item(), torch.distributions.Uniform(1,2).sample((1,1)).item(), torch.distributions.Uniform(1,2).sample((1,1)).item()
# v = lambda x,y: a*(((x+1)/2)**4 - 6*((x+1)/2)**2*((y+1)/2)**2 + ((y+1)/2)**4) + b*((x+1)/2)**4 + c*((y+1)/2)**4

# v1 = lambda y: 3*a*(y+1)**2/4 + 3*c*(y+1)**2
# v2 = lambda y: -3*a*(4-(y+1)**2)/4 + 3*c*(y+1)**2

# v3 = lambda x: 3*a*(x+1)**2/4 + 3*b*(x+1)**2
# v4 = lambda x: 3*a*((x+1)**2-4)/4 + 3*b*(x+1)**2

# f = lambda x,y: 12*(c*(y+1)**2 + b*(x+1)**2)/4 + \
#     (-(1-x)/2*v1(y) - (1+x)/2*v2(y) - (1-y)/2*v3(x) - (1+y)/2*v4(x))


# w = lambda x, y: (
#     - (1 - x) / 2 * v(torch.tensor(-1), y)                        # 上邊界 x = -1
#     - (1 + x) / 2 * v(torch.tensor( 1), y)                        # 下邊界 x = 1
#     - (1 - y) / 2 * v(x, torch.tensor(-1))                        # 左邊界 y = -1
#     - (1 + y) / 2 * v(x, torch.tensor( 1))                        # 右邊界 y = 1
#     + (1 - x) * (1 - y) / 4 * v(torch.tensor(-1), torch.tensor(-1))             # 左上角
#     + (1 - x) * (1 + y) / 4 * v(torch.tensor(-1), torch.tensor( 1))             # 右上角
#     + (1 + x) * (1 - y) / 4 * v(torch.tensor( 1), torch.tensor(-1))             # 左下角
#     + (1 + x) * (1 + y) / 4 * v(torch.tensor( 1), torch.tensor( 1))             # 右下角
# )

# u = lambda x, y: v(x,y) + w(x,y)
from test_functions import TEST_CASES

# u, f = TEST_CASES["multimode"]
# 或
u, f = TEST_CASES["polynomial"]
# u = lambda x, y: torch.sin(x*torch.pi) * torch.sin(y*torch.pi)
# f = lambda x, y: -2*torch.pi**2 * torch.sin(x*torch.pi) * torch.sin(y*torch.pi)
with torch.no_grad():
    ((X2d, Y2d), (wx, wy), W2d, D, D2, U, F), train_data, train_data_xy, train_data_st, train_label_f, train_label_u = u_train_data_generator(Ngrid, 1)
    train_data_xy = train_data_xy.to(device)
    train_data_st = train_data_st.to(device)
    train_data    = train_data.to(device)
    train_label_f = train_label_f.to(device)
    train_label_u = train_label_u.to(device)
    
    loaded_model = load_model(None, f'{model_folder}/models', f"model_single.pth", device=f'{device}', load_entire=True)
    if net_dim[0] == 2:
        G_pred = loaded_model(train_data_xy, train_data_st).reshape(Ngrid+1, Ngrid+1, Ngrid+1, Ngrid+1).cpu()
    elif net_dim[0] == 4:
        G_pred = loaded_model(train_data).reshape(Ngrid+1, Ngrid+1, Ngrid+1, Ngrid+1).cpu()
    
    # r = (G_pred - train_label.reshape(Ngrid+1, Ngrid+1, Ngrid+1, Ngrid+1).cpu()).abs()  
    # G_Linf.append(torch.max(r).item())

    U = u(X2d, Y2d)
    F = D2@U +U@D2.T
    u_pred = torch.einsum('ijkl,kl->ij', G_pred, W2d * F)
    u_residual_exact = u_pred - U
    err_exact_inf = torch.max(torch.abs(u_residual_exact))
    u_Linf_exact.append(err_exact_inf.item())
    print(f'Maximum Error (vs exact): {err_exact_inf.item():.2e}')

    err_exact_2 = torch.norm(u_residual_exact, 2) / torch.norm(U, 2)
    u_L2_exact.append(err_exact_2.item())
    print(f'Relative L2 Error (vs exact): {err_exact_2.item():.2e}')
    
    fig = plt.figure(figsize=(16, 8))
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(X2d.cpu(), Y2d.cpu(), U.cpu(), alpha=0.6)
    ax1.set_title('exact u')

    ax2 = fig.add_subplot(122, projection='3d')
    ax2.plot_surface(X2d.cpu(), Y2d.cpu(), u_pred.cpu(), alpha=0.6)
    ax2.set_title('preidict u')